# MERGE FILE

In [65]:
# 1. Import Library yang dibutuhkan
import pandas as pd
import glob
import os


In [66]:
# 2. Hubungkan ke Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [67]:
# 3. Ambil semua file CSV dalam folder
path = '/content/drive/MyDrive/Cyclistic-Bike-Share-Analysis/cyclistic-data-2025'
all_files = glob.glob(os.path.join(path, "*.csv"))

In [68]:
# 4. Verifikasi Integritas Kolom (Memastikan 12 file memiliki struktur yang sama)
print("--- Verifikasi Struktur Kolom ---")
for f in all_files:
    cols = pd.read_csv(f, nrows=0).columns.tolist()
    print(f"{os.path.basename(f)}: {len(cols)} kolom - OK")

--- Verifikasi Struktur Kolom ---
202501-divvy-tripdata.csv: 13 kolom - OK
202502-divvy-tripdata.csv: 13 kolom - OK
202503-divvy-tripdata.csv: 13 kolom - OK
202504-divvy-tripdata.csv: 13 kolom - OK
202505-divvy-tripdata.csv: 13 kolom - OK
202506-divvy-tripdata.csv: 13 kolom - OK
202507-divvy-tripdata.csv: 13 kolom - OK
202508-divvy-tripdata.csv: 13 kolom - OK
202509-divvy-tripdata.csv: 13 kolom - OK
202510-divvy-tripdata.csv: 13 kolom - OK
202511-divvy-tripdata.csv: 13 kolom - OK
202512-divvy-tripdata.csv: 13 kolom - OK


In [69]:
# 5. Gabungkan 12 file menjadi satu
df = pd.concat((pd.read_csv(f) for f in all_files), ignore_index=True)
initial_rows = len(df)

In [70]:
print(f"Jumlah file yang ditemukan: {len(all_files)}")
print(f"Total kolom setelah digabung: {len(df.columns):,}")
#df.head()

Jumlah file yang ditemukan: 12
Total kolom setelah digabung: 13


# Tahap Pembersihan (Process)

In [71]:
# Memastikan setiap perjalanan hanya tercatat satu kali
df = df.drop_duplicates(subset=['ride_id'])

In [72]:
# Menghapus baris yang tidak memiliki Nama Stasiun (Krusial untuk Analisis Rute)
df_clean = df.dropna(subset=['start_station_name', 'end_station_name']).copy()

In [73]:
# Menghapus spasi berlebih di awal/akhir nama stasiun agar tidak terdobel di Dashboard
df_clean['start_station_name'] = df_clean['start_station_name'].str.strip()
df_clean['end_station_name'] = df_clean['end_station_name'].str.strip()

In [74]:
# TRANSFORMASI WAKTU & DURASI
df_clean['started_at'] = pd.to_datetime(df_clean['started_at'])
df_clean['ended_at'] = pd.to_datetime(df_clean['ended_at'])

In [75]:
# Hitung ride_length dalam menit
df_clean['ride_length'] = (df_clean['ended_at'] - df_clean['started_at']).dt.total_seconds() / 60

In [76]:
# Hitung day_of_week (1=Minggu, 7=Sabtu sesuai Roadmap)
df_clean['day_of_week'] = df_clean['started_at'].dt.dayofweek + 2
df_clean.loc[df_clean['day_of_week'] == 8, 'day_of_week'] = 1

In [77]:
# Hapus durasi <= 0 dan data pemeliharaan sistem
df_clean = df_clean[df_clean['ride_length'] > 0]
df_clean = df_clean[~df_clean['start_station_name'].str.contains("TEST|HQ QR", na=False)]

In [78]:
#Hapus kolom koordinat yang memperberat ukuran file
df_clean = df_clean.drop(columns=['start_lat', 'start_lng', 'end_lat', 'end_lng'])

# Ringkasan Akhir untuk Doc.Github

In [79]:
final_rows = len(df_clean)
rows_removed = initial_rows - final_rows

print(f"\n--- AUDIT PEMBERSIHAN DATA ---")
print(f"Total Data Mentah   : {initial_rows:,}")
print(f"Data Bersih Akhir   : {final_rows:,}")
print(f"Total Data Dihapus  : {rows_removed:,} (Duplikat, Null, Error, Test)")

# Simpan ke CSV Final
df_clean.to_csv('cyclistic_final_clean.csv', index=False)


--- AUDIT PEMBERSIHAN DATA ---
Total Data Mentah   : 5,552,994
Data Bersih Akhir   : 3,693,235
Total Data Dihapus  : 1,859,759 (Duplikat, Null, Error, Test)
